# PROT-ON: structure-based detection of designer PROTein interface MutatiONs

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CSB-KaracaLab/prot-on/blob/main/PROT-ON.ipynb)

PROT-ON takes the structure of a protein dimer, mutates every interface residue of one chain to the other 19 amino acids, and scores each mutant with [EvoEF1](https://github.com/tommyhuangthu/EvoEF) (bundled) or [FoldX](http://foldxsuite.crg.eu/) (your own licensed copy). It then reports the mutations that are predicted to enrich or deplete binding, filtered by their effect on the stability of the scanned chain.

**How to use this notebook**

1. Run the cells from top to bottom (in Colab: *Runtime → Run all*). With the default settings this runs the example complex.
2. To scan your own dimer, change the settings in step 2 and run steps 2 to 4 again.

The example (532 mutations) takes about a minute on a laptop. Larger interfaces take proportionally longer. The notebook runs on Google Colab, Linux and macOS.

If you use PROT-ON, please cite: Koşaca M, Yılmazbilek İ, Karaca E (2023). PROT-ON: A structure-based detection of designer PROTein interface MutatiONs. *Frontiers in Molecular Biosciences* 10:1063971. [doi:10.3389/fmolb.2023.1063971](https://doi.org/10.3389/fmolb.2023.1063971)

## 1. Set up PROT-ON
Downloads PROT-ON (unless the notebook is already inside a copy of the repository), compiles EvoEF1 and installs the plotting packages.

In [ ]:
#@title Set up PROT-ON { display-mode: "form" }
import os, shutil, subprocess, sys
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/CSB-KaracaLab/prot-on.git"


def _find_repo():
    here = Path.cwd().resolve()
    for folder in (here, *here.parents):
        if (folder / "proton.py").is_file() and (folder / "EvoEF1" / "src").is_dir():
            return folder
    return None


REPO = _find_repo()
if REPO is None:
    if not Path("prot-on/proton.py").is_file():
        subprocess.run(["git", "clone", "--quiet", "--depth", "1", REPO_URL, "prot-on"], check=True)
    REPO = Path("prot-on").resolve()
os.chdir(REPO)
(REPO / "results").mkdir(exist_ok=True)

# EvoEF1 is shipped as source code and compiled here.
EVOEF = REPO / "EvoEF1" / "EvoEF"
if not EVOEF.is_file():
    if shutil.which("g++") is None:
        raise RuntimeError("A C++ compiler (g++) is needed to build EvoEF1, and none was found.")
    build = subprocess.run("g++ -O3 -ffast-math -w -o EvoEF src/*.cpp", shell=True,
                           cwd=REPO / "EvoEF1", capture_output=True, text=True)
    if build.returncode != 0 or not EVOEF.is_file():
        print(build.stderr[-3000:])
        raise RuntimeError("EvoEF1 could not be compiled.")


def _major(package):
    try:
        return int(version(package).split(".")[0])
    except PackageNotFoundError:
        return None


# PROT-ON saves its figures with plotly 5 and kaleido 0.2; newer kaleido needs a Chrome installation.
wanted = [name for name in ("numpy", "pandas", "py3Dmol") if _major(name) is None]
if _major("plotly") != 5 or _major("kaleido") != 0:
    wanted += ["plotly==5.24.1", "kaleido==0.2.1"]
if wanted:
    print("Installing:", ", ".join(wanted))
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", *wanted], check=True)

RUNS = {}                                     # every scan of this session, keyed by structure and chain
print("PROT-ON is ready in", REPO)

## 2. Input and settings

**Structure.** PROT-ON needs a PDB file with exactly two chains. Keep `example-input/complex.pdb` to try the example. To scan your own dimer, tick `upload_pdb` (Colab), or put the path *or the web address* of a PDB file in `pdb_file`. A docking model works as well as an experimental structure: a HADDOCK model, for example, can be scanned straight from its result page by pasting the address of its PDB file.

A chain of the file can hold more than one polypeptide. An antibody whose heavy and light chains were both given the chain ID `A` is scanned as one molecule, so the whole paratope is covered in a single run.

The structure is read by the first of the two cells below, which then prints the chains it found and offers them in a **Chain to scan** dropdown, with the number of residues of each. The chain you pick there is the one whose interface residues are mutated; changing it and running step 3 again scans the other chain, with nothing else to re-run.

**Settings**, in the second cell.
- `cut_off`: two residues are at the interface if any of their atoms are within this distance (Å).
- `IQR`: multiplier of the interquartile range used to call the outliers (the enriching and depleting mutations) of the ΔΔG distribution.
- `algorithm`: `EvoEF1` works out of the box. `FoldX` needs your own academic-licensed Linux `foldx` executable and `rotabase.txt`; you will be asked to upload them. The FoldX route is experimental in this notebook.

**Optional PSSM filter.** To also filter the mutations by evolutionary information, provide the PSSM of the scanned chain as a comma-separated file (one row per residue of the chain, one column per amino acid), for example from [POSSUM](https://possum.erc.monash.edu/server.jsp). Tick `upload_pssm`, or type its path in `pssm_file`. For the example complex you can use `example-input/complex_chain_D_pssm.csv`.

**Following the HADDOCK antibody–antigen tutorial?** The cell below fetches the three structures used in its PROT-ON bonus task. Tick its box, run it, and it prints the paths to paste into `pdb_file`; the antibody is chain `A` in all three.

In [ ]:
#@title Optional: structures of the HADDOCK antibody–antigen tutorial { display-mode: "form" }
download_tutorial_structures = False #@param {type:"boolean"}
#@markdown The interleukin-1β / gevokizumab complex of the
#@markdown [HADDOCK2.4 antibody–antigen tutorial](https://www.bonvinlab.org/education/HADDOCK24/HADDOCK24-antibody-antigen-basic/#bonus-3-dissecting-the-interface-energetics-prot-on-analysis):
#@markdown the crystal structure and two docking models of it, to be scanned one after another.
import io, urllib.request, zipfile

TUTORIAL = {
    "4G6M-matched.pdb": ("https://www.bonvinlab.org/education/HADDOCK24/HADDOCK24-antibody-antigen-basic/input-data.zip",
                         "HADDOCK24-antibody-antigen-basic/4G6M-matched.pdb", "crystal structure"),
    "cluster5_1.pdb": ("https://wenmr.science.uu.nl/haddock2.4/run/4242424242/184871-4G6M-Ab-Ag-NMR-Ab-no-passive/cluster5_1.pdb",
                       None, "best model of the top-ranked cluster"),
    "cluster2_1.pdb": ("https://wenmr.science.uu.nl/haddock2.4/run/4242424242/184871-4G6M-Ab-Ag-NMR-Ab-no-passive/cluster2_1.pdb",
                       None, "best model of the cluster with the lowest ligand-RMSD"),
}

if download_tutorial_structures:
    folder = REPO / "notebook_inputs" / "haddock_tutorial"
    folder.mkdir(parents=True, exist_ok=True)
    archives = {}
    for name, (url, member, description) in TUTORIAL.items():
        target = folder / name
        if not target.is_file():
            if member is None:
                with urllib.request.urlopen(url, timeout=120) as response:
                    target.write_bytes(response.read())
            else:
                if url not in archives:
                    with urllib.request.urlopen(url, timeout=120) as response:
                        archives[url] = zipfile.ZipFile(io.BytesIO(response.read()))
                target.write_bytes(archives[url].read(member))
        print(f"{target.relative_to(REPO).as_posix():52s} {description}")
    print("\nCopy one of these paths into pdb_file in the cell below; the antibody is chain A in all three.")
else:
    print("Tick the box to download the structures of the HADDOCK antibody-antigen tutorial.")

In [ ]:
#@title Structure and chain selection { display-mode: "form" }
pdb_file = "example-input/complex.pdb" #@param {type:"string"}
upload_pdb = False #@param {type:"boolean"}
#@markdown A PDB file with exactly two chains: the path of one, its web address, or tick `upload_pdb`
#@markdown to choose a file from your computer. Running the cell prints the chains it found and
#@markdown offers them in a dropdown; pick the one to scan there.
import re
import urllib.request

os.chdir(REPO)
INPUTS = REPO / "notebook_inputs"


def _upload(what):
    """Ask for one file in Colab and return (name, content)."""
    if not IN_COLAB:
        raise RuntimeError(f"Uploading works only in Google Colab. Type the path of {what} instead.")
    from google.colab import files
    print(f"Choose {what}:")
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError(f"No file was uploaded for {what}.")
    name, content = next(iter(uploaded.items()))
    for leftover in uploaded:                 # Colab also writes the files into the working folder
        Path(leftover).unlink(missing_ok=True)
    return name, content


def _safe_stem(name):
    stem = re.sub(r"[^A-Za-z0-9_-]+", "_", Path(name).stem).strip("_")
    return stem or "complex"


# --- structure -------------------------------------------------------------
if upload_pdb:
    name, content = _upload("your PDB file")
    INPUTS.mkdir(exist_ok=True)
    PDB_PATH = INPUTS / (_safe_stem(name) + ".pdb")
    PDB_PATH.write_bytes(content)
elif re.match(r"https?://", pdb_file.strip()):
    url = pdb_file.strip()
    INPUTS.mkdir(exist_ok=True)
    PDB_PATH = INPUTS / (_safe_stem(url.split("?")[0].rstrip("/").split("/")[-1]) + ".pdb")
    print("Downloading", url)
    with urllib.request.urlopen(url, timeout=180) as response:
        PDB_PATH.write_bytes(response.read())
else:
    source = Path(pdb_file.strip()).expanduser()
    if not source.is_file():
        raise FileNotFoundError(f"PDB file not found: {pdb_file}")
    source = source.resolve()
    inside_repo = REPO in source.parents
    relative = source.relative_to(REPO).as_posix() if inside_repo else ""
    if inside_repo and re.fullmatch(r"[A-Za-z0-9_./-]+\.pdb", relative):
        PDB_PATH = source
    else:                                     # PROT-ON needs a simple file name ending in .pdb
        INPUTS.mkdir(exist_ok=True)
        PDB_PATH = INPUTS / (_safe_stem(source.name) + ".pdb")
        shutil.copyfile(source, PDB_PATH)
PDB_NAME = PDB_PATH.stem

# --- the chains of the file, offered as a selection ------------------------
residues = {}
with open(PDB_PATH) as handle:
    for line in handle:
        if line[:4] == "ATOM":
            residues.setdefault(line[21], set()).add(line[22:26])
if not residues:
    raise ValueError(f"{PDB_PATH.name} holds no ATOM records. PROT-ON needs a structure in the PDB format; "
                     "an mmCIF file has to be converted first.")


def _chain_label(chain):
    return f"{chain if chain.strip() else '(blank)'} ({len(residues[chain])} residues)"


CHAIN_OPTIONS = list(residues)
chain_summary = ", ".join(_chain_label(chain) for chain in CHAIN_OPTIONS)
if len(CHAIN_OPTIONS) != 2:
    raise ValueError(f"PROT-ON works only with dimers, but {PDB_PATH.name} has {len(CHAIN_OPTIONS)} chains: "
                     f"{chain_summary}. Please keep only the two chains of interest in the file.")
STARTS_ON = {"example-input/complex.pdb": "D"}   # the chain the published example output was made with
chain_ID = STARTS_ON.get(pdb_file.strip(), CHAIN_OPTIONS[0])
if chain_ID not in CHAIN_OPTIONS:
    chain_ID = CHAIN_OPTIONS[0]

CHAIN_SELECTOR = None
try:                                          # a dropdown of the chains found in the file
    import ipywidgets
    from IPython.display import display
    CHAIN_SELECTOR = ipywidgets.Dropdown(options=[(_chain_label(c), c) for c in CHAIN_OPTIONS],
                                         value=chain_ID, description="Chain to scan:",
                                         style={"description_width": "initial"})
except ImportError:                           # a plain Python session: chain_ID decides
    pass


def chain_to_scan():
    """The chain picked in the dropdown, or chain_ID when there is no dropdown."""
    chain = CHAIN_SELECTOR.value if CHAIN_SELECTOR is not None else chain_ID
    if chain not in residues:
        raise ValueError(f"Chain '{chain}' is not in {PDB_PATH.name}. Its chains are: {chain_summary}.")
    loaded_pssm = globals().get("pssm")       # only once the next cell has read one
    if loaded_pssm is not None and len(loaded_pssm) != len(residues[chain]):
        raise ValueError(f"The PSSM has {len(loaded_pssm)} rows, but chain {chain} has "
                         f"{len(residues[chain])} residues. It needs one row per residue of the chain you scan.")
    return chain


print(f"Structure : {PDB_PATH.name} with chains {chain_summary}")
if CHAIN_SELECTOR is not None:
    print("\nChoose the chain to scan below, then run the next cell.")
    display(CHAIN_SELECTOR)
else:
    print(f"\nipywidgets is not installed, so there is no dropdown and chain {chain_ID} will be scanned. "
          f'To scan the other one, run  chain_ID = "{next(c for c in CHAIN_OPTIONS if c != chain_ID)}"  '
          "in a cell of your own before step 3.")

In [ ]:
#@title PROT-ON parameters { display-mode: "form" }
cut_off = 5.0 #@param {type:"number"}
IQR = 1.5 #@param {type:"number"}
algorithm = "EvoEF1" #@param ["EvoEF1", "FoldX"]
pssm_file = "" #@param {type:"string"}
upload_pssm = False #@param {type:"boolean"}
#@markdown `cut_off`: two residues are at the interface when any of their atoms are this close (Å).
#@markdown `IQR`: how far outside the interquartile range a score has to fall to count as an outlier.
#@markdown `algorithm`: EvoEF1 is bundled; FoldX needs your own licensed copy. The PSSM file is optional.
import pandas as pd

AMINO_ACIDS = list("ACDEFGHIKLMNPQRSTVWY")
if cut_off <= 0 or IQR <= 0:
    raise ValueError("cut_off and IQR must be positive numbers.")

# --- optional PSSM ---------------------------------------------------------
PSSM_PATH, pssm = None, None
if upload_pssm or pssm_file.strip():
    if upload_pssm:
        INPUTS.mkdir(exist_ok=True)
        name, content = _upload("the PSSM file (.csv)")
        PSSM_PATH = INPUTS / "uploaded_pssm.csv"
        PSSM_PATH.write_bytes(content)
    else:
        PSSM_PATH = Path(pssm_file.strip()).expanduser().resolve()
        if not PSSM_PATH.is_file():
            raise FileNotFoundError(f"PSSM file not found: {pssm_file}")
    pssm = pd.read_csv(PSSM_PATH, sep=",")
    missing = [aa for aa in AMINO_ACIDS if aa not in pssm.columns]
    if missing:
        raise ValueError("The PSSM file must be comma separated with one column per amino acid. "
                         f"Missing columns: {', '.join(missing)}")

# --- optional FoldX --------------------------------------------------------
if algorithm == "FoldX":
    for filename in ("foldx", "rotabase.txt"):
        target = REPO / "src" / filename
        if not target.is_file():
            if not IN_COLAB:
                raise FileNotFoundError(f"To use FoldX, place '{filename}' in {REPO / 'src'}")
            _, content = _upload(f"your FoldX file '{filename}'")
            target.write_bytes(content)
    (REPO / "src" / "foldx").chmod(0o755)

print(f"Scanning : chain {chain_to_scan()} of {PDB_PATH.name} with {algorithm} "
      f"(cut-off {cut_off} Å, IQR {IQR})")
print(f"PSSM     : {PSSM_PATH.name if PSSM_PATH else 'not used'}")

## 3. Run PROT-ON
Builds every interface mutant, scores it, and runs the statistical analysis. The line below the cell shows the progress.

When it is done it offers the results as a zip file: the score tables, the lists of enriching and depleting mutations, the figures (png and svg), the run log and, unless you untick it, the structure of every mutant. In Colab the download starts when you press the button. Nothing is packed until you ask for it, and the results stay in the folder the cell prints either way.

In [ ]:
#@title Run PROT-ON { display-mode: "form" }
import signal, tempfile, threading, time

os.chdir(REPO)
chain_ID = chain_to_scan()                    # whatever is selected in the dropdown of step 2
results_root = REPO / "results"
prefix = f"{PDB_NAME}_chain_{chain_ID}"
output_name = f"{prefix}_{algorithm}_output"


def _work_folders():
    return {p for p in REPO.iterdir() if p.is_dir() and re.fullmatch(r"[0-9a-f]{32}", p.name)}


def _count_lines(path):
    with open(path) as handle:
        return sum(1 for line in handle if line.strip())


results_before = {p.name for p in results_root.iterdir()}
work_before = _work_folders()
root_copy = REPO / PDB_PATH.name              # proton.py leaves a copy of the input here
had_root_copy = root_copy.exists()
pssm_copy = None
if PSSM_PATH is not None:                     # the PSSM filter looks for this file name next to the scripts
    pssm_copy = REPO / "src" / f"{prefix}_pssm.csv"
    shutil.copyfile(PSSM_PATH, pssm_copy)

# PROT-ON starts its scripts with "python"; make sure that is this notebook's Python.
shim = Path(tempfile.mkdtemp(prefix="proton_python_"))
(shim / "python").write_text(f'#!/bin/sh\nexec "{sys.executable}" "$@"\n')
(shim / "python").chmod(0o755)
env = dict(os.environ, PYTHONUNBUFFERED="1", PATH=f"{shim}{os.pathsep}{os.environ.get('PATH', '')}")

command = [sys.executable, "proton.py", "--pdb", PDB_PATH.relative_to(REPO).as_posix(),
           "--chain_ID", chain_ID, "--algorithm", algorithm, "--cut_off", str(cut_off), "--IQR", str(IQR)]
log_path = REPO / "proton_run.log"
progress = {"stage": "Starting", "pass": 0, "built": 0, "contacts": 0}
STAGES = {"Defining the Interface": "Defining the interface",
          "Energies are being": "Computing binding and stability energies",
          "Outlier Detection Process": "Analysing the scores",
          "Box plot analysis": "Drawing the figures"}


def _follow(stream):
    with open(log_path, "w") as log:
        for line in stream:
            log.write(line)
            if "Distance is" in line:
                progress["contacts"] += 1
                continue
            built = re.match(r"EvoEF Building Mutation Model (\d+)", line)
            if built:
                number = int(built.group(1))
                if number == 1:
                    progress["pass"] += 1
                progress.update(stage="Building the mutant models", built=number)
                continue
            for marker, stage in STAGES.items():
                if line.startswith(marker):
                    progress["stage"] = stage


def _status(work, total, started):
    stage = progress["stage"]
    detail = ""
    scored = 0
    if work is not None:
        scored = len(list(work.glob("*_output/mutation_models/*.pdb"))) + len(list(work.glob("src/mutation_models/*.pdb")))
    of_total = f" of {total}" if total else ""
    if stage == "Building the mutant models":
        detail = f": {progress['built']}{of_total} ({'complex' if progress['pass'] < 2 else 'scanned chain alone'})"
    elif scored:
        if stage in ("Starting", "Defining the interface"):
            stage = "Building and scoring the mutants"
        detail = f": {scored}{of_total} mutations scored"
    minutes, seconds = divmod(int(time.time() - started), 60)
    return f"{stage}{detail} | {minutes:02d}:{seconds:02d} elapsed"


started = time.time()
process = subprocess.Popen(command, cwd=REPO, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                           text=True, errors="replace", start_new_session=True)
reader = threading.Thread(target=_follow, args=(process.stdout,), daemon=True)
reader.start()
RESULT_DIR = None
try:
    work, total = None, None
    while process.poll() is None:
        time.sleep(1)
        if work is None:
            work = next(iter(_work_folders() - work_before), None)
        if work is not None and total is None:
            for candidate in (work / "src" / f"{prefix}_mutation_list", work / "EvoEF1" / f"{prefix}_mutation_list",
                              work / "EvoEF1" / "individual_list.txt"):
                if candidate.is_file():
                    total = _count_lines(candidate) or None
                    break
        print("\r" + _status(work, total, started).ljust(100), end="", flush=True)
    reader.join(timeout=10)
    print()
    new_results = [p for p in results_root.iterdir() if p.is_dir() and p.name not in results_before]
    finished = [p / output_name for p in new_results if (p / output_name / f"{prefix}_proton_scores").is_file()]
    if not finished:
        if progress["contacts"] == 0:
            print(f"No atoms of the two chains are within {cut_off} Å of each other, so there is no interface "
                  "to scan. Check the structure, or increase cut_off.\n")
        with open(log_path) as log:
            tail = [line.rstrip() for line in log if "Distance is" not in line][-25:]
        print("\n".join(tail))
        raise RuntimeError(f"PROT-ON did not finish. The last lines of its log are shown above; the full log is in {log_path}")
    RESULT_DIR = finished[0]
    shutil.move(str(log_path), str(RESULT_DIR / "proton_run.log"))
finally:
    if process.poll() is None:                # interrupted: stop PROT-ON and everything it started
        os.killpg(process.pid, signal.SIGTERM)
        process.wait()
    for folder in _work_folders() - work_before:
        shutil.rmtree(folder, ignore_errors=True)
    if pssm_copy is not None:
        pssm_copy.unlink(missing_ok=True)
    if not had_root_copy:
        root_copy.unlink(missing_ok=True)
    shutil.rmtree(shim, ignore_errors=True)

TABLE_NAMES = ("proton_scores", "enriching_mutations", "depleting_mutations",
               "stabilizing_enriching_mutations", "stabilizing_depleting_mutations",
               "pssm_enriching", "pssm_depleting")
RUN_LABEL = f"{PDB_NAME} chain {chain_ID}"
RUNS[RUN_LABEL] = {
    "result_dir": RESULT_DIR, "prefix": prefix, "algorithm": algorithm, "ddg": f"DDG_{algorithm}_Scores",
    "cut_off": cut_off, "IQR": IQR,
    "tables": {name: (pd.read_csv(RESULT_DIR / f"{prefix}_{name}", sep=" ")
                      if (RESULT_DIR / f"{prefix}_{name}").is_file() else None) for name in TABLE_NAMES},
}

tables = RUNS[RUN_LABEL]["tables"]
scores = tables["proton_scores"]
ddg = RUNS[RUN_LABEL]["ddg"]


def download_results(include_mutation_models=True):
    """Pack this run's results folder into a zip file; in Colab the download then starts."""
    import zipfile
    run_folder = RESULT_DIR.parent
    archive = run_folder.with_suffix(".zip")
    with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as bundle:
        for path in sorted(RESULT_DIR.rglob("*")):
            if path.is_file() and (include_mutation_models or "mutation_models" not in path.parts):
                bundle.write(path, Path(run_folder.name) / path.relative_to(RESULT_DIR))
    print(f"Saved {archive} ({archive.stat().st_size / 1e6:.1f} MB)")
    if IN_COLAB:
        from google.colab import files
        files.download(str(archive))
    return archive


minutes, seconds = divmod(int(time.time() - started), 60)
print(f"PROT-ON finished in {minutes} min {seconds} s.")
print("Results folder:", RESULT_DIR)

try:                                          # the zip holds the tables, the figures, the log and the mutants
    import ipywidgets
    from IPython.display import display
    with_models = ipywidgets.Checkbox(value=True, description="include the mutant structures", indent=False)
    button = ipywidgets.Button(description="Download the results", icon="download")
    log = ipywidgets.Output()

    def _download(_):
        button.disabled = True
        with log:
            download_results(with_models.value)
        button.disabled = False

    button.on_click(_download)
    print("\nDownload the results of this run?")
    display(ipywidgets.HBox([button, with_models]), log)
except ImportError:
    print("\nTo save the results as a zip file, run  download_results()  in a cell of your own.")

## 4. Results

**The two ends of the scan in 3D.** The cell below draws the mutant PROT-ON predicts to bind best next to the one it predicts to bind worst, each one in the interface it was built in: the partner chain as a solid surface, the scanned chain as a ribbon, its interface residues as thin sticks and the mutated side chain as thick ones. Drag to rotate, scroll to zoom. The mutant structures are EvoEF1's own models, the ones its scores were computed from, so this is a look at what was scored rather than a new model.

In [ ]:
#@title The best and the worst mutant in 3D { display-mode: "form" }
designer_mutations_only = True #@param {type:"boolean"}
surface_opacity = 0.65 #@param {type:"slider", min:0.2, max:1.0, step:0.05}
#@markdown With `designer_mutations_only` ticked, the two mutants are taken from the designer tables; untick it to
#@markdown take the lowest and the highest ΔΔG of the whole scan, outliers or not.
import py3Dmol
from IPython.display import Markdown, display

MODELS = RESULT_DIR / "mutation_models"
INTERFACE = []
interface_list = RESULT_DIR / f"{prefix}_interface_aa_list"
if interface_list.is_file():
    for line in interface_list.read_text().splitlines():
        parts = line.split()
        if len(parts) == 3 and parts[2].lstrip("-").isdigit():
            INTERFACE.append(int(parts[2]))

PICKED_FROM = {
    "stabilizing_enriching_mutations": "enriching designer mutation (an outlier that also stabilizes the chain)",
    "enriching_mutations": "enriching outlier (none of them stabilize the chain)",
    "stabilizing_depleting_mutations": "depleting designer mutation (an outlier that also stabilizes the chain)",
    "depleting_mutations": "depleting outlier (none of them stabilize the chain)",
    "proton_scores": "mutation of the scan, outlier or not",
}


def _strongest(names, lowest):
    """The strongest mutation of the first of these tables that holds one, else of the whole scan."""
    for name in names:
        table = RUNS[RUN_LABEL]["tables"].get(name)
        if table is not None and not table.empty:
            return table.loc[table[ddg].idxmin() if lowest else table[ddg].idxmax()], name
    return scores.loc[scores[ddg].idxmin() if lowest else scores[ddg].idxmax()], "proton_scores"


def _model_of(row):
    """The mutant structure EvoEF1 or FoldX built for this mutation."""
    position = row["Positions"]
    code = f"{position[0]}{chain_ID}{position[1:]}{row['Mutations']}"
    found = sorted(MODELS.glob(f"*{code}.pdb")) if MODELS.is_dir() else []
    return found[0] if found else None


def _atoms(text):
    return [line for line in text.splitlines() if line[:4] == "ATOM" and len(line) > 54]


def _centre(text, number):
    """Where to hang the label: the CA of the mutated residue, or any of its atoms."""
    for wanted in ("CA", ""):
        for line in _atoms(text):
            if line[21] == chain_ID and line[22:26].strip() == str(number) and line[12:16].strip().startswith(wanted):
                return {"x": float(line[30:38]), "y": float(line[38:46]), "z": float(line[46:54])}
    return None


def _xyz(text, chain, number=None):
    picked = []
    for line in _atoms(text):
        if line[21] != chain or (number is not None and line[22:26].strip() != str(number)):
            continue
        picked.append((line[22:26].strip(), float(line[30:38]), float(line[38:46]), float(line[46:54])))
    return picked


def _touched(text, partner, number):
    """The residues of the partner chain the mutated side chain reaches, at PROT-ON's own cut-off."""
    limit = RUNS[RUN_LABEL]["cut_off"] ** 2
    mutated = _xyz(text, chain_ID, number)
    return sorted({int(resi) for resi, x, y, z in _xyz(text, partner)
                   if any((x - mx) ** 2 + (y - my) ** 2 + (z - mz) ** 2 <= limit for _, mx, my, mz in mutated)})


enriching = ["stabilizing_enriching_mutations", "enriching_mutations"] if designer_mutations_only else []
depleting = ["stabilizing_depleting_mutations", "depleting_mutations"] if designer_mutations_only else []
panels = [(*_strongest(enriching, True), "Best", "green"),
          (*_strongest(depleting, False), "Worst", "red")]

view = py3Dmol.view(width=900, height=420, viewergrid=(1, 2), linked=False)
rows, missing = [], []
for column, (row, source, title, colour) in enumerate(panels):
    mutation = f"{row['Positions']}{row['Mutations']}"
    path = _model_of(row)
    reached = "no model was kept"
    if path is None:
        missing.append(mutation)
    else:
        text = path.read_text()
        cell = (0, column)
        number = int(row["Positions"][1:])
        partner = next((c for c in sorted({line[21] for line in _atoms(text)}) if c != chain_ID), chain_ID)
        touched = _touched(text, partner, number)
        reached = f"{len(touched)} residues of chain {partner}"
        elsewhere = [resi for resi in sorted({int(r) for r, *_ in _xyz(text, partner)}) if resi not in set(touched)]
        view.addModel(text, "pdb", viewer=cell)
        view.setStyle({"chain": partner}, {"cartoon": {"color": "white"}}, viewer=cell)
        view.addSurface(py3Dmol.VDW, {"opacity": surface_opacity, "color": "lightgrey"},
                        {"chain": partner, "resi": elsewhere}, viewer=cell)
        if touched:                           # the patch of the partner this very side chain reaches
            view.addSurface(py3Dmol.VDW, {"opacity": min(1.0, surface_opacity + 0.25), "color": "gold"},
                            {"chain": partner, "resi": touched}, viewer=cell)
        view.setStyle({"chain": chain_ID}, {"cartoon": {"color": "lightsteelblue"}}, viewer=cell)
        if INTERFACE:
            view.addStyle({"chain": chain_ID, "resi": INTERFACE},
                          {"stick": {"radius": 0.1, "color": "steelblue"}}, viewer=cell)
        view.addStyle({"chain": chain_ID, "resi": number}, {"stick": {"radius": 0.3, "color": colour}}, viewer=cell)
        spot = _centre(text, number)
        if spot is not None:
            view.addLabel(f"{mutation}   ΔΔG {row[ddg]:+.2f}",
                          {"position": spot, "backgroundColor": colour, "backgroundOpacity": 0.85,
                           "fontColor": "white", "fontSize": 12}, viewer=cell)
        view.zoomTo(viewer=cell)

    rows.append(f"| **{title}** | {mutation} | {row[ddg]:+.2f} | {row['DDG_Stability_Scores']:+.2f} | "
                f"strongest {PICKED_FROM[source]} | {reached} |")

display(Markdown("\n".join(["| | mutation | ΔΔG binding | ΔΔG stability | picked as the | side chain reaches |",
                            "|---|---|---|---|---|---|"] + rows)))
if missing:
    print("No mutant structure was kept for: " + ", ".join(missing))
print(f"Left: the best mutant, right: the worst. The partner chain is the surface, gold where this side chain "
      f"touches it; chain {chain_ID} is the ribbon, its interface residues drawn as thin sticks.")
view.show()

In [ ]:
#@title Heatmap of every scanned mutation (hover to read the values) { display-mode: "form" }
import re
import plotly.express as px
from IPython.display import Image, Markdown, display

matrix = scores.pivot_table(index="Positions", columns="Mutations", values=ddg)
order = sorted(matrix.index, key=lambda position: int(re.search(r"-?\d+", position).group()))
figure = px.imshow(matrix.loc[order], color_continuous_scale="RdBu_r", color_continuous_midpoint=0,
                   aspect="auto", labels=dict(x="Mutation", y="Position", color=f"{algorithm} ΔΔG"))
figure.update_layout(width=750, height=max(400, 22 * len(order) + 150),
                     title=f"Chain {chain_ID} of {PDB_NAME}: all scanned mutations")
figure.show()

saturated = RESULT_DIR / f"{prefix}_heatmap.png"
if saturated.is_file():
    display(Markdown("#### The same scores as PROT-ON draws them, with the outliers saturated"))
    display(Image(filename=str(saturated), width=700))

In [ ]:
#@title Box plots { display-mode: "form" }
from IPython.display import Image, Markdown, display

figures = {"ResidueSpecificBoxPlot": "ΔΔG scores per mutant amino acid",
           "PositionSpecificBoxPlot": "ΔΔG scores per position",
           "boxplot": "Distribution of all ΔΔG scores"}
for name, title in figures.items():
    path = RESULT_DIR / f"{prefix}_{name}.png"
    if path.is_file():
        display(Markdown(f"#### {title}"))
        display(Image(filename=str(path), width=700))

**Reading the tables.** `DDG_<algorithm>_Scores` is the binding score of the mutant minus that of the wild type: negative values are predicted to enrich binding, positive values to deplete it. `DDG_Stability_Scores` is the same difference for the stability of the scanned chain alone: negative values are stabilizing. The *designer mutations* are the enriching and depleting outliers that are also stabilizing.

In [ ]:
#@title Designer mutations { display-mode: "form" }
from IPython.display import Image, Markdown, display

if RESULT_DIR is None:
    raise RuntimeError("Run step 3 first.")

display(Markdown(
    f"**{len(scores)} mutations** were scanned at **{scores['Positions'].nunique()} interface positions** "
    f"of chain {chain_ID}. Binding-enriching outliers: **{len(tables['enriching_mutations'])}** "
    f"({len(tables['stabilizing_enriching_mutations'])} also stabilizing). "
    f"Binding-depleting outliers: **{len(tables['depleting_mutations'])}** "
    f"({len(tables['stabilizing_depleting_mutations'])} also stabilizing)."))

titles = {"stabilizing_enriching_mutations": "Designer mutations: binding enriching and stabilizing",
          "stabilizing_depleting_mutations": "Designer mutations: binding depleting and stabilizing",
          "pssm_enriching": "Enriching designer mutations that also pass the PSSM filter",
          "pssm_depleting": "Depleting designer mutations that also pass the PSSM filter"}
for name, title in titles.items():
    table = tables[name]
    if table is None:
        continue
    display(Markdown(f"#### {title}"))
    if table.empty:
        print("None found.")
    else:
        display(table.sort_values(ddg).reset_index(drop=True))

## 5. Experimental: re-score the designer mutations with HADDOCK3

PROT-ON scans every interface mutation with EvoEF1 because EvoEF1 is fast. A different question is whether a second energy function, with the mutant actually refined rather than modelled in place, agrees about which mutations matter. [HADDOCK3](https://github.com/haddocking/haddock3) can answer that, and `pip install haddock3` ships the CNS binaries, so there is nothing to request. HADDOCK3 prints the terms at every start and they are worth repeating: CNS is free for non-profit use, and a commercial user is responsible for holding a licence.

The mutation itself is done by HADDOCK. Each mutated residue is cut back to its backbone and CB and renamed, which leaves the side chain as *missing atoms*; `topoaa` then rebuilds it (`build-missing.cns`) and the refinement settles it. That rebuild starts from a randomly perturbed position, so unlike EvoEF1 this is **not deterministic**: the same mutation refined twice gives two slightly different numbers, which is why `replicas` below is worth more than one. The spread across replicas is the honest error bar, and a ΔΔG difference smaller than it means nothing.

The whole shortlist goes through **one** HADDOCK3 run as an ensemble, not one run per mutation. Even so this is far slower than EvoEF1, so it is pointed at the designer mutations rather than the whole scan. Scoring the complex alone is not a binding energy, so the mutated chain is refined and scored on its own as well, and the partner chain cancels in the difference:

ΔΔG = [score(complex, mutant) − score(chain, mutant)] − [score(complex, wild type) − score(chain, wild type)]

**This has never been run.** It is written against the HADDOCK3 source rather than tested, it needs Linux (the macOS CNS binary wants Homebrew's gcc), and the first thing it prints is how long a model took, which decides whether scanning a whole interface this way is sensible at all.

In [ ]:
#@title Install HADDOCK3 { display-mode: "form" }
install_haddock3 = False #@param {type:"boolean"}
#@markdown A few minutes. The wheel carries a statically linked CNS for Linux, so nothing
#@markdown else is needed on Colab.
if install_haddock3:
    import shutil as _shutil
    if _shutil.which("haddock3") is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "haddock3"], check=True)
    out = subprocess.run(["haddock3", "--version"], capture_output=True, text=True)
    print(out.stdout.strip() or out.stderr.strip())
    from haddock.core.defaults import cns_exec
    print("CNS:", cns_exec, "| present:", Path(str(cns_exec)).is_file())
else:
    print("Tick the box to install HADDOCK3.")

In [ ]:
#@title Re-score with HADDOCK3 { display-mode: "form" }
which_mutations = "designer" #@param ["designer", "enriching outliers", "depleting outliers"]
max_mutations = 12 #@param {type:"integer"}
replicas = 3 #@param {type:"integer"}
em_steps = 200 #@param {type:"integer"}
ncores = 2 #@param {type:"integer"}
#@markdown Start small. `max_mutations` x `replicas` x 2 structures is what gets refined, and
#@markdown the cell prints the time per model so you can decide what a full scan would cost.
import re, signal, threading, time

import pandas as pd
from IPython.display import Markdown, display

THREE = {"A": "ALA", "C": "CYS", "D": "ASP", "E": "GLU", "F": "PHE", "G": "GLY", "H": "HIS",
         "I": "ILE", "K": "LYS", "L": "LEU", "M": "MET", "N": "ASN", "P": "PRO", "Q": "GLN",
         "R": "ARG", "S": "SER", "T": "THR", "V": "VAL", "W": "TRP", "Y": "TYR"}
BACKBONE = {"N", "CA", "C", "O", "CB", "OXT"}


def _is_hydrogen(line):
    element = line[76:78].strip()
    return element == "H" if element else line[12:16].strip().lstrip("0123456789").startswith("H")


def _mutate(lines, chain, number, target, only_chain=None):
    """Cut the residue back to backbone and CB and rename it; HADDOCK rebuilds the rest."""
    keep = BACKBONE - ({"CB"} if THREE[target] == "GLY" else set())
    out = []
    for line in lines:
        if line[:4] != "ATOM" or _is_hydrogen(line):
            continue
        if only_chain is not None and line[21] != only_chain:
            continue
        if line[21] == chain and line[22:26].strip() == str(number):
            if line[12:16].strip() not in keep:
                continue
            line = line[:17] + f"{THREE[target]:>3}" + line[20:]
        out.append(line)
    return out


def _ensemble(path, lines, chosen, chain, only_chain=None):
    """Wild type first, then one model per mutation, in the order of `chosen`."""
    with open(path, "w") as fh:
        for index, row in enumerate([None] + chosen, start=1):
            fh.write(f"MODEL     {index:>4}\n")
            if row is None:
                fh.writelines(_mutate(lines, chain, -1, "A", only_chain))   # no residue matches -1
            else:
                fh.writelines(_mutate(lines, chain, row[0], row[1], only_chain))
            fh.write("ENDMDL\n")
        fh.write("END\n")
    return path


def _scores(step_dir):
    """The tsv a scoring module writes: one row per refined model."""
    for tsv in sorted(Path(step_dir).glob("*.tsv")):
        table = pd.read_csv(tsv, sep="\t")
        if "score" in table.columns:
            return table
    raise FileNotFoundError(f"no scoring table in {step_dir}")


def _by_input_model(table):
    """Average the replicas of each starting model, keyed by its ensemble index."""
    index = table["original_name"].astype(str).str.extract(r"_(\d+)")[0].astype(int)
    return table.assign(model=index).groupby("model")["score"].agg(["mean", "std", "count"])


if RESULT_DIR is None:
    raise RuntimeError("Run step 3 first: this re-scores mutations PROT-ON has already found.")
if shutil.which("haddock3") is None:
    raise RuntimeError("HADDOCK3 is not installed: run the cell above.")

source = {"designer": ["stabilizing_enriching_mutations", "stabilizing_depleting_mutations"],
          "enriching outliers": ["enriching_mutations"],
          "depleting outliers": ["depleting_mutations"]}[which_mutations]
frames = [tables[name] for name in source
          if tables.get(name) is not None and not tables[name].empty]
if not frames:
    raise RuntimeError(f"this run found no {which_mutations}; try another set")
picked = pd.concat(frames)
if picked.empty:
    raise RuntimeError(f"this run found no {which_mutations}; try another set")
picked = picked.reindex(picked[ddg].abs().sort_values(ascending=False).index).head(max_mutations)
chosen = [(int(row["Positions"][1:]), row["Mutations"]) for _, row in picked.iterrows()]
print(f"re-scoring {len(chosen)} mutations with {replicas} replicas each:",
      ", ".join(f"{p}{m}" for p, m in zip(picked['Positions'], picked['Mutations'])))

work = REPO / "haddock3_rescoring"
shutil.rmtree(work, ignore_errors=True)
(work).mkdir(parents=True)
atoms = PDB_PATH.read_text().splitlines(keepends=True)
_ensemble(work / "complex_ensemble.pdb", atoms, chosen, chain_ID)
_ensemble(work / "chain_ensemble.pdb", atoms, chosen, chain_ID, only_chain=chain_ID)

for name in ("complex", "chain"):
    (work / f"{name}.toml").write_text(f"""
run_dir = "{name}_run"
mode = "local"
ncores = {ncores}
molecules = ["{name}_ensemble.pdb"]

[topoaa]

[emref]
nemsteps = {em_steps}
sampling_factor = {replicas}
tolerance = 20

[emscoring]
""".lstrip())

def _run_haddock(name):
    """Run one workflow, showing which module is working and how far it has got."""
    state = {"module": "starting", "total": 0, "done": 0}
    log_path = work / f"{name}.log"
    process = subprocess.Popen(["haddock3", f"{name}.toml"], cwd=work, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True, errors="replace",
                               start_new_session=True)

    def follow(stream):
        with open(log_path, "w") as log:
            for line in stream:
                log.write(line)
                log.flush()
                module = re.search(r"\] \[(\w+)\]", line)
                if module and module.group(1) != state["module"]:
                    state.update(module=module.group(1), total=0, done=0)
                jobs = re.search(r"Running CNS Jobs n=(\d+)", line)
                if jobs:
                    state["total"] = int(jobs.group(1))
                finished = re.search(r"(\d+) tasks finished", line)
                if finished:
                    state["done"] = int(finished.group(1))

    reader = threading.Thread(target=follow, args=(process.stdout,), daemon=True)
    reader.start()
    begun = time.time()
    try:
        while process.poll() is None:
            time.sleep(1)
            minutes, seconds = divmod(int(time.time() - begun), 60)
            of = f" {state['done']}/{state['total']} jobs" if state["total"] else ""
            print(f"\r  {state['module']}{of} | {minutes:02d}:{seconds:02d} elapsed".ljust(68),
                  end="", flush=True)
        reader.join(timeout=5)
        print()
    finally:
        if process.poll() is None:            # interrupted: stop CNS and everything it started
            os.killpg(process.pid, signal.SIGTERM)
            process.wait()
    return process.returncode


started = time.time()
for name in ("complex", "chain"):
    print(f"\nHADDOCK3 on the {name} ensemble: {len(chosen) + 1} structures x {replicas} replicas")
    if _run_haddock(name) != 0:
        with open(work / f"{name}.log") as log:
            tail = [line.rstrip() for line in log if line.strip()][-25:]
        print("\n".join(tail))
        raise RuntimeError(f"HADDOCK3 failed on the {name} ensemble. Its last lines are above; the full "
                           f"log is {work / (name + '.log')} and the run directory {work / (name + '_run')}")
models = (len(chosen) + 1) * replicas * 2
elapsed = time.time() - started
print(f"\n{models} models refined and scored in {elapsed / 60:.1f} min "
      f"({elapsed / models:.1f} s per model)")
print(f"a 532-mutation scan at this rate would take {532 * 2 * replicas * elapsed / models / 3600:.1f} h")

step = {name: next(p for p in sorted((work / f"{name}_run").iterdir()) if "emscoring" in p.name)
        for name in ("complex", "chain")}
complex_scores = _by_input_model(_scores(step["complex"]))
chain_scores = _by_input_model(_scores(step["chain"]))

binding = complex_scores["mean"] - chain_scores["mean"]
spread = (complex_scores["std"].fillna(0) ** 2 + chain_scores["std"].fillna(0) ** 2) ** 0.5
result = pd.DataFrame({
    "Mutation": [f"{p}{m}" for p, m in zip(picked["Positions"], picked["Mutations"])],
    f"DDG_{algorithm}": picked[ddg].to_numpy(),
    "DDG_HADDOCK3": [binding.get(i + 2, float("nan")) - binding.get(1, float("nan"))
                     for i in range(len(chosen))],
    "replica_spread": [spread.get(i + 2, float("nan")) for i in range(len(chosen))],
})
display(Markdown("#### The same mutations, scored two ways"))
display(result.round(2))

usable = result.dropna()
if len(usable) > 2:
    ranks = usable[[f"DDG_{algorithm}", "DDG_HADDOCK3"]].rank()
    rho = ranks.corr().iloc[0, 1]
    print(f"rank correlation between the two scores over {len(usable)} mutations: {rho:.2f}")
    print(f"median replica spread: {usable['replica_spread'].median():.2f} "
          "(a difference smaller than this is noise)")